In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import nmslib
from tqdm.auto import tqdm

from rectools import Columns
from rectools.metrics import MAP
from rectools.dataset import Dataset
from rectools.models import model_from_config

In [2]:
RANDOM_SEED = 42
os.environ["OPENBLAS_NUM_THREADS"] = "1"
warnings.filterwarnings("ignore")

In [3]:
DATA_PATH = Path("data_original")
interactions = pd.read_csv(DATA_PATH / "interactions.csv", parse_dates=["last_watch_dt"])

interactions = interactions.rename(columns={"total_dur": Columns.Weight, "last_watch_dt": Columns.Datetime})
users = pd.read_csv(DATA_PATH / "users.csv")
items = pd.read_csv(DATA_PATH / "items.csv")

In [4]:
N_DAYS = 7
max_date = interactions[Columns.Datetime].max()
train = interactions[interactions[Columns.Datetime] <= max_date - pd.Timedelta(days=N_DAYS)]
test = interactions[interactions[Columns.Datetime] > max_date - pd.Timedelta(days=N_DAYS)]

test_users = test[Columns.User].unique()
cold_users = set(test_users) - set(train[Columns.User])
test = test[~test[Columns.User].isin(cold_users)]
hot_users = test[Columns.User].unique()

In [5]:
K_RECOS = 10
map10 = MAP(k=K_RECOS)

### Preprocessing

In [6]:
user_features_frames = []
for feature in ["sex", "age", "income"]:
    feature_frame = users[[Columns.User, feature]].copy()
    feature_frame.columns = ["id", "value"]
    feature_frame["feature"] = feature
    user_features_frames.append(feature_frame)

user_features = pd.concat(user_features_frames)

In [7]:
items = items.loc[items[Columns.Item].isin(train[Columns.Item])].copy()

In [8]:
items["genre"] = items["genres"].str.lower().str.replace(", ", ",", regex=False).str.split(",")
genre_feature = items[[Columns.Item, "genre"]].explode("genre")
genre_feature.columns = ["id", "value"]
genre_feature["feature"] = "genre"

In [9]:
content_feature = items[[Columns.Item, "content_type"]].copy()
content_feature.columns = ["id", "value"]
content_feature["feature"] = "content_type"

In [10]:
item_features = pd.concat([genre_feature, content_feature])

In [11]:
def process_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    reference_date = df_copy[Columns.Datetime].max()
    df_copy["recency"] = (reference_date - df_copy[Columns.Datetime]).dt.days

    df_copy["log_total_dur"] = np.log1p(df_copy[Columns.Weight])

    df_copy["watched_pct_norm"] = df_copy["watched_pct"] / 100.0

    df_copy["watch_month"] = df_copy[Columns.Datetime].dt.month
    df_copy["watch_dayofweek"] = df_copy[Columns.Datetime].dt.dayofweek

    user_agg = (
        df_copy.groupby("user_id")
        .agg(
            total_dur_sum=(Columns.Weight, "sum"),
            total_dur_mean=(Columns.Weight, "mean"),
            recency_mean=("recency", "mean"),
            interactions_count=("user_id", "count"),
        )
        .reset_index()
    )

    item_agg = (
        df_copy.groupby("item_id")
        .agg(
            total_dur_sum=(Columns.Weight, "sum"),
            watched_pct_mean=("watched_pct", "mean"),
            interactions_count=("item_id", "count"),
        )
        .reset_index()
    )

    df_copy = df_copy.merge(user_agg, on="user_id", how="left", suffixes=("", "_user"))
    df_copy = df_copy.merge(item_agg, on="item_id", how="left", suffixes=("", "_item"))

    df_copy.drop(columns=["watched_pct", Columns.Weight], inplace=True)
    df_copy.rename(columns={"log_total_dur": Columns.Weight}, inplace=True)

    return df_copy

In [12]:
train2 = process_dataframe(train)

In [13]:
dataset_with_features = Dataset.construct(
    interactions_df=train2,
    user_features_df=user_features,
    cat_user_features=["sex", "age", "income"],
    item_features_df=item_features,
    cat_item_features=["genre", "content_type"],
)

### LightFM

In [14]:
config = {
    "cls": "LightFMWrapperModel",
    "model": {
        "no_components": 32,
        "learning_rate": 0.05,
        "user_alpha": 0,
        "item_alpha": 0,
        "loss": "warp",
        "random_state": RANDOM_SEED,
    },
    "epochs": 1,
}

In [102]:
model = model_from_config(config)
model.fit(dataset_with_features)

recos = model.recommend(
    users=hot_users,
    dataset=dataset_with_features,
    k=K_RECOS,
    filter_viewed=True,
)
print("Standard LightFM MAP@10:", map10.calc(recos, test))

Standard LightFM MAP@10: 0.09170830159633923


### ANN

In [16]:
user_embeddings, item_embeddings = model.get_vectors(dataset_with_features)

In [17]:
user_embeddings.shape, item_embeddings.shape

((1042716, 34), (15577, 34))

In [ ]:
external_user_ids = dataset_with_features.user_id_map.external_ids
external_item_ids = dataset_with_features.item_id_map.external_ids

user_to_index = {uid: idx for idx, uid in enumerate(external_user_ids)}
user_id_to_model = {hot_user: user_to_index[hot_user] for hot_user in hot_users}

item_model_to_id = {i: external_item_ids[i] for i in range(len(external_item_ids))}

In [23]:
def recommend_all(query_factors, index_factors, topn=10):
    output = query_factors.dot(index_factors.T)
    argpartition_indices = np.argpartition(output, -topn)[:, -topn:]

    x_indices = np.repeat(np.arange(output.shape[0]), topn)
    y_indices = argpartition_indices.flatten()
    top_value = output[x_indices, y_indices].reshape(output.shape[0], topn)
    top_indices = np.argsort(top_value)[:, ::-1]

    y_indices = top_indices.flatten()
    top_indices = argpartition_indices[x_indices, y_indices]
    labels = top_indices.reshape(-1, topn)
    distances = output[x_indices, top_indices].reshape(-1, topn)
    return labels, distances

In [95]:
def recommend_index(index, user_ids) -> pd.DataFrame:
    rows = []

    for user_id in tqdm(user_ids):
        user_model_id = user_id_to_model[user_id]
        nbsr = index.knnQueryBatch(user_embeddings[[user_model_id], :], k=10)
        recs, distances = nbsr[0]
        for rank, (item, score) in enumerate(zip(recs, distances), start=1):
            rows.append({"user_id": user_id, "item_id": item_model_to_id[item], "score": score, "rank": rank})

    return pd.DataFrame(rows)

In [101]:
index_time_params = {
    "M": 72,
    "efConstruction": 200,
    "post": 2,
}

index = nmslib.init(method="hnsw", space="negdotprod", data_type=nmslib.DataType.DENSE_VECTOR)
index.addDataPointBatch(item_embeddings)
index.createIndex(index_time_params)

index.setQueryTimeParams({"efSearch": 200})

ann_recos = recommend_index(index, hot_users)
print("ANN LightFM MAP@10:", map10.calc(ann_recos, test))

ANN LightFM MAP@10: 0.08878979708234327


### Results

In [ ]:
user_items = ann_recos.groupby("user_id")["item_id"].apply(list).to_dict()

In [ ]:
import json

with open("reco_hot_lightfm_ann.json", "w") as f:
    json.dump(user_items, f)